# Chapter 7 Source Code Notebook

This notebook builds the source code for Chapter 7, **Memory as a First-Class System Component**.

The code builds working, episodic, and semantic memory layers with retrieval checks, freshness rules, and trace output for SupportOps AI.

Before running the notebook, install any required packages and set the required API keys in your environment where model powered examples are used.


## Step 1: Memory Entry Schema

### `memory/schema.py`

The memory layer starts with a schema so every stored item has consistent structure. This makes memory easier to rank, filter, retrieve, and audit later.

In [ ]:
# memory/schema.py
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Optional

class MemoryType(Enum):
    WORKING  = "working"
    EPISODIC = "episodic"
    SEMANTIC = "semantic"
    STRUCTURED = "structured"

class SourceType(Enum):
    VERIFIED_DB    = "verified_db"      # from trusted database
    OFFICIAL_DOC   = "official_doc"     # from official documentation
    MODEL_SYNTHESIS = "model_synthesis"  # generated by a model
    USER_INPUT     = "user_input"        # from customer message
    COMMUNITY      = "community"         # from forum / unreviewed

@dataclass
class MemoryEntry:
    entry_id:           str
    memory_type:        MemoryType
    content:            str
    source_type:        SourceType
    created_at:         str = field(
        default_factory=lambda: datetime.utcnow().isoformat()
    )
    updated_at:         str = field(
        default_factory=lambda: datetime.utcnow().isoformat()
    )
    confidence:         float = 1.0     # 0.0 - 1.0
    authority_score:    float = 1.0     # 0.0 - 1.0
    superseded_by:      Optional[str] = None
    embedding:          Optional[list] = None
    metadata:           dict = field(default_factory=dict)
    retrieval_count:    int = 0
    last_retrieved_at:  Optional[str] = None

    @property
    def is_active(self) -> bool:
        return self.superseded_by is None

    @property
    def age_days(self) -> float:
        created = datetime.fromisoformat(self.created_at)
        return (datetime.utcnow() - created).total_seconds() / 86400


## Step 2: Embedding Service

### `memory/embeddings.py`

After the schema is in place, the embedding service turns text into vectors for semantic comparison. This prepares the system to retrieve meaningfully related information, not just exact keyword matches.

In [ ]:
# memory/embeddings.py
import hashlib
from typing import Optional

class EmbeddingService:
    """
    Embedding interface. Default implementation uses a simple
    hash-based mock for testing. In production, replace with
    your vector provider (OpenAI, Cohere, sentence-transformers).
    """
    def __init__(self, provider: str = "mock"):
        self.provider = provider
        self._cache: dict[str, list[float]] = {}

    def embed(self, text: str) -> list[float]:
        if text in self._cache:
            return self._cache[text]
        if self.provider == "mock":
            vec = self._mock_embed(text)
        else:
            vec = self._api_embed(text)
        self._cache[text] = vec
        return vec

    def similarity(self, a: list[float],
                    b: list[float]) -> float:
        """Cosine similarity between two vectors."""
        if not a or not b:
            return 0.0
        dot   = sum(x*y for x,y in zip(a,b))
        mag_a = sum(x*x for x in a) ** 0.5
        mag_b = sum(x*x for x in b) ** 0.5
        if mag_a == 0 or mag_b == 0:
            return 0.0
        return dot / (mag_a * mag_b)

    def _mock_embed(self, text: str) -> list[float]:
        """Deterministic mock - for testing only."""
        h = hashlib.sha256(text.lower().encode()).digest()
        return [b / 255.0 for b in h[:32]]

    def _api_embed(self, text: str) -> list[float]:
        # Replace with your provider
        raise NotImplementedError("Configure embedding provider")


## Step 3: Semantic Memory with Hybrid Retrieval

### `memory/semantic.py`

Semantic memory then combines vector similarity with additional ranking signals. The goal is to retrieve useful knowledge while still keeping relevance and freshness visible.

In [ ]:
# memory/semantic.py
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional
from .schema import MemoryEntry, MemoryType, SourceType
from .embeddings import EmbeddingService

@dataclass
class RetrievalResult:
    entry:            MemoryEntry
    semantic_score:   float
    keyword_score:    float
    freshness_score:  float
    authority_score:  float
    combined_score:   float

@dataclass
class RetrievalWeights:
    semantic:   float = 0.45
    keyword:    float = 0.20
    freshness:  float = 0.20
    authority:  float = 0.15

    def __post_init__(self):
        total = (self.semantic + self.keyword +
                 self.freshness + self.authority)
        assert abs(total - 1.0) < 0.001

class SemanticMemory:
    def __init__(self,
                 embeddings: EmbeddingService,
                 weights: RetrievalWeights = None,
                 relevance_threshold: float = 0.35,
                 freshness_half_life_days: float = 90.0):
        self.embeddings   = embeddings
        self.weights      = weights or RetrievalWeights()
        self.threshold    = relevance_threshold
        self.half_life    = freshness_half_life_days
        self._entries: dict[str, MemoryEntry] = {}

    def store(self, content: str,
               source_type: SourceType = SourceType.OFFICIAL_DOC,
               confidence: float = 1.0,
               authority_score: float = 1.0,
               metadata: dict = None,
               supersedes: Optional[str] = None) -> MemoryEntry:
        entry_id = str(uuid.uuid4())
        entry = MemoryEntry(
            entry_id      = entry_id,
            memory_type   = MemoryType.SEMANTIC,
            content       = content,
            source_type   = source_type,
            confidence    = confidence,
            authority_score = authority_score,
            embedding     = self.embeddings.embed(content),
            metadata      = metadata or {}
        )
        if supersedes and supersedes in self._entries:
            self._entries[supersedes].superseded_by = entry_id
        self._entries[entry_id] = entry
        return entry

    def retrieve(self, query: str,
                  top_k: int = 3) -> list[RetrievalResult]:
        query_vec = self.embeddings.embed(query)
        query_lower = query.lower()
        results = []
        for entry in self._entries.values():
            if not entry.is_active:
                continue
            # Semantic score
            sem = self.embeddings.similarity(
                query_vec, entry.embedding or []
            )
            # Keyword score: fraction of query words in content
            words = [w for w in query_lower.split() if len(w) > 3]
            kw = (sum(1 for w in words if w in entry.content.lower())
                  / max(len(words), 1))
            # Freshness score: exponential decay
            import math
            fresh = math.exp(
                -entry.age_days * math.log(2) / self.half_life
            )
            # Authority score: from entry
            auth = entry.authority_score * entry.confidence
            # Combined
            w = self.weights
            combined = (w.semantic   * sem  +
                        w.keyword    * kw   +
                        w.freshness  * fresh +
                        w.authority  * auth)
            if combined >= self.threshold:
                results.append(RetrievalResult(
                    entry=entry, semantic_score=sem,
                    keyword_score=kw, freshness_score=fresh,
                    authority_score=auth, combined_score=combined
                ))
        results.sort(key=lambda r: r.combined_score, reverse=True)
        # Update retrieval tracking
        for r in results[:top_k]:
            r.entry.retrieval_count += 1
            r.entry.last_retrieved_at = datetime.utcnow().isoformat()
        return results[:top_k]


## Step 4: Episodic Memory with Summarization

### `memory/episodic.py`

Episodic memory adds customer and ticket history to the system. Summaries keep prior interactions usable without flooding the context window.

In [ ]:
# memory/episodic.py
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional
import anthropic
from .schema import MemoryEntry, MemoryType, SourceType

@dataclass
class EpisodeRecord:
    episode_id:    str
    customer_id:   str
    ticket_id:     str
    outcome:       str   # resolved | escalated | failed
    category:      str
    sentiment:     str
    escalated:     bool
    commitments:   list[str] = field(default_factory=list)
    summary:       str = ""
    created_at:    str = field(
        default_factory=lambda: datetime.utcnow().isoformat()
    )

SUMMARIZE_PROMPT = """
Summarize this customer support interaction history into 3-5 sentences.
Include: main issues raised, outcomes, any commitments made,
overall sentiment trend, and any unresolved items.
Be factual and concise. Omit pleasantries and operational detail.

History:
{history}
"""

class EpisodicMemory:
    def __init__(self, client: anthropic.Anthropic = None,
                 max_full_episodes: int = 3,
                 summarize_model: str = "claude-haiku-4-5-20251001"):
        self.client             = client
        self.max_full           = max_full_episodes
        self.summarize_model    = summarize_model
        self._episodes: dict[str, list[EpisodeRecord]] = {}
        # customer_id -> list of episodes, newest first

    def store(self, customer_id: str,
               episode: EpisodeRecord) -> None:
        if customer_id not in self._episodes:
            self._episodes[customer_id] = []
        self._episodes[customer_id].insert(0, episode)

    def get_context(self, customer_id: str) -> dict:
        """
        Returns a context dict suitable for model injection:
        - recent: last max_full episodes in full
        - summary: summarized older episodes (if any)
        - stats: aggregate statistics
        """
        episodes = self._episodes.get(customer_id, [])
        if not episodes:
            return {"recent": [], "summary": None, "stats": {}}
        recent = episodes[:self.max_full]
        older  = episodes[self.max_full:]
        summary = None
        if older and self.client:
            history = "\n".join([
                f"[{e.created_at[:10]}] {e.outcome}: {e.summary or e.category}"
                for e in older
            ])
            try:
                resp = self.client.messages.create(
                    model=self.summarize_model,
                    max_tokens=256,
                    messages=[{
                        "role": "user",
                        "content": SUMMARIZE_PROMPT.format(history=history)
                    }]
                )
                summary = resp.content[0].text.strip()
            except Exception:
                summary = f"{len(older)} older interactions (summary unavailable)"
        stats = {
            "total_interactions": len(episodes),
            "escalation_rate": sum(
                1 for e in episodes if e.escalated
            ) / max(len(episodes), 1),
            "last_outcome": episodes[0].outcome if episodes else None
        }
        return {
            "recent": [
                {"date": e.created_at[:10], "outcome": e.outcome,
                 "category": e.category, "escalated": e.escalated,
                 "summary": e.summary or e.category}
                for e in recent
            ],
            "summary": summary,
            "stats": stats
        }


## Step 5: Memory Trace

### `memory/trace.py`

The memory trace records what was retrieved and why it was selected. This makes memory influence visible instead of letting retrieved context silently shape the response.

In [ ]:
# memory/trace.py
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional
from .semantic import RetrievalResult

@dataclass
class MemoryTraceEntry:
    session_id:       str
    step_name:        str
    query:            str
    retrieved_count:  int
    results:          list[dict]  # {entry_id, score, content_preview}
    threshold_used:   float
    blocked_count:    int = 0     # results below threshold
    used_in_context:  bool = True
    timestamp:        str = field(
        default_factory=lambda: datetime.utcnow().isoformat()
    )
    coverage_score:   float = 0.0  # 0-1: does retrieved content
                                    # cover query intent?

class MemoryTracer:
    def __init__(self, session_id: str):
        self.session_id = session_id
        self._entries: list[MemoryTraceEntry] = []

    def record_retrieval(self,
                          step_name: str,
                          query: str,
                          results: list[RetrievalResult],
                          threshold: float,
                          blocked: int = 0) -> MemoryTraceEntry:
        coverage = (
            sum(r.combined_score for r in results) / max(len(results), 1)
            if results else 0.0
        )
        entry = MemoryTraceEntry(
            session_id      = self.session_id,
            step_name       = step_name,
            query           = query,
            retrieved_count = len(results),
            results         = [
                {
                    "entry_id": r.entry.entry_id,
                    "combined_score": round(r.combined_score, 3),
                    "semantic": round(r.semantic_score, 3),
                    "freshness": round(r.freshness_score, 3),
                    "content_preview": r.entry.content[:80]
                }
                for r in results
            ],
            threshold_used  = threshold,
            blocked_count   = blocked,
            coverage_score  = coverage
        )
        self._entries.append(entry)
        return entry

    def coverage_report(self) -> dict:
        """Aggregated retrieval quality across all steps."""
        if not self._entries:
            return {"steps": 0}
        return {
            "steps":           len(self._entries),
            "avg_coverage":    sum(e.coverage_score
                               for e in self._entries) / len(self._entries),
            "zero_result_steps": sum(
                1 for e in self._entries if e.retrieved_count == 0
            ),
            "low_coverage_steps": sum(
                1 for e in self._entries if e.coverage_score < 0.4
            ),
            "total_blocked":   sum(e.blocked_count for e in self._entries)
        }


## Step 6: Context Assembler

### `memory/assembler.py`

The context assembler turns retrieved memory into the final information package for reasoning. It controls ordering and limits so the most useful evidence is easier for the model to use.

In [ ]:
# memory/assembler.py
from dataclasses import dataclass
from typing import Optional
from .semantic import SemanticMemory
from .episodic import EpisodicMemory
from .trace import MemoryTracer

@dataclass
class AssembledContext:
    working_summary:   str
    episodic_context:  dict
    retrieved_docs:    list[dict]
    total_tokens_est:  int
    memory_trace:      list[dict]

def estimate_tokens(text: str) -> int:
    """Rough estimate: 1 token ~ 4 characters."""
    return len(text) // 4

class ContextAssembler:
    def __init__(self,
                 semantic:  SemanticMemory,
                 episodic:  EpisodicMemory,
                 max_tokens: int = 8000):
        self.semantic   = semantic
        self.episodic   = episodic
        self.max_tokens = max_tokens

    def assemble(self,
                  session_id:   str,
                  customer_id:  Optional[str],
                  query:        str,
                  step_name:    str,
                  tracer:       MemoryTracer) -> AssembledContext:
        budget = self.max_tokens
        # 1. Episodic context (customer history)
        episodic = {}
        if customer_id:
            episodic = self.episodic.get_context(customer_id)
            episodic_text = str(episodic)
            budget -= estimate_tokens(episodic_text)
        # 2. Semantic retrieval (knowledge base)
        top_k = 3
        raw_results = self.semantic.retrieve(query, top_k=top_k + 3)
        # Filter by token budget
        selected, blocked = [], 0
        for r in raw_results:
            doc_tokens = estimate_tokens(r.entry.content)
            if doc_tokens <= budget:
                selected.append(r)
                budget -= doc_tokens
                if len(selected) >= top_k:
                    break
            else:
                blocked += 1
        # 3. Record memory trace
        tracer.record_retrieval(
            step_name, query, selected,
            self.semantic.threshold, blocked
        )
        retrieved_docs = [
            {"content": r.entry.content,
             "source": r.entry.source_type.value,
             "score": round(r.combined_score, 3),
             "entry_id": r.entry.entry_id}
            for r in selected
        ]
        return AssembledContext(
            working_summary  = f"Session: {session_id}",
            episodic_context = episodic,
            retrieved_docs   = retrieved_docs,
            total_tokens_est = self.max_tokens - budget,
            memory_trace     = [
                {"entry_id": r.entry.entry_id,
                 "score": round(r.combined_score, 3)}
                for r in selected
            ]
        )


## Step 7: Seeding the Knowledge Base and Running a Query

### `memory/run_memory.py`

The final cell seeds sample memory and runs a query through the retrieval flow. It shows how schema, embeddings, semantic memory, episodic memory, tracing, and context assembly connect.

In [ ]:
# memory/run_memory.py
import os
import anthropic
from supportops_ai.memory.schema import SourceType
from supportops_ai.memory.embeddings import EmbeddingService
from supportops_ai.memory.semantic import SemanticMemory, RetrievalWeights
from supportops_ai.memory.episodic import EpisodicMemory, EpisodeRecord
from supportops_ai.memory.trace import MemoryTracer
from supportops_ai.memory.assembler import ContextAssembler

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
embeddings = EmbeddingService(provider="mock")  # swap for real provider

# Build memory layers
semantic  = SemanticMemory(embeddings,
    weights=RetrievalWeights(semantic=0.45, keyword=0.20,
                             freshness=0.20, authority=0.15),
    relevance_threshold=0.35)

episodic  = EpisodicMemory(client=client, max_full_episodes=3)
assembler = ContextAssembler(semantic, episodic, max_tokens=6000)

# Seed knowledge base
doc_return = semantic.store(
    "Return Policy v2.4 (effective 2024-01-01): Customers may return products within 30 days of purchase for a full refund. Digital products are non-refundable. Extended 60-day returns apply to Enterprise tier.",
    source_type=SourceType.OFFICIAL_DOC,
    confidence=1.0, authority_score=1.0,
    metadata={"doc_id": "POL-001", "version": "2.4"}
)

doc_billing = semantic.store(
    "Duplicate Charge Resolution: If a customer is charged twice for the same invoice, verify via the billing portal before issuing a refund. Refunds process within 3-5 business days. Do not commit to a specific timeline without checking current processing queue.",
    source_type=SourceType.OFFICIAL_DOC,
    confidence=1.0, authority_score=0.9,
    metadata={"doc_id": "FAQ-042"}
)

# Seed episodic memory for a repeat customer
episodic.store("C-4421", EpisodeRecord(
    episode_id="EP-001", customer_id="C-4421",
    ticket_id="TKT-001", outcome="resolved",
    category="billing", sentiment="negative",
    escalated=False, commitments=["refund_issued"],
    summary="Customer disputed a duplicate charge. Refund of $24.99 issued."
))

# Assemble context for a new query
tracer = MemoryTracer("TKT-009")
context = assembler.assemble(
    session_id  = "TKT-009",
    customer_id = "C-4421",
    query       = "I was charged twice and want a refund",
    step_name   = "draft_response",
    tracer      = tracer
)

print("=== ASSEMBLED CONTEXT ===")
print(f"Est tokens:    {context.total_tokens_est}")
print(f"Retrieved docs: {len(context.retrieved_docs)}")
for doc in context.retrieved_docs:
    print(f"  [{doc['score']}] {doc['content'][:70]}...")
print(f"\nEpisodic history:")
print(f"  Recent interactions: {len(context.episodic_context.get('recent', []))}")
print(f"  Last outcome: {context.episodic_context.get('stats', {}).get('last_outcome')}")

print("\n=== MEMORY TRACE COVERAGE REPORT ===")
report = tracer.coverage_report()
print(f"  Steps:              {report['steps']}")
print(f"  Avg coverage:       {report['avg_coverage']:.2f}")
print(f"  Zero-result steps:  {report['zero_result_steps']}")
print(f"  Low-coverage steps: {report['low_coverage_steps']}")
print(f"  Total blocked:      {report['total_blocked']}")
